# Gradients Training Demo

In this notebook, we take a general Qwen model and teach it how to answer biomedical research questions using PubMedQA medical data.

The journey is simple: give Gradients a dataset, let the network train the model, then test it on medical questions it has never seen before. At the end, you will see the base model and trained model side by side, with the expected answer from the test set.

No training scripts, no infrastructure work, no ML ops setup. Just an API key, a dataset, and a few notebook cells.

## Step 1: Install The Gradients SDK

One install gives this notebook everything it needs to launch training, sample data, load models, merge adapters, and run inference.

In [ ]:
%pip install -q --upgrade gradientsio==0.1.2

## Step 2: Choose The Mission

We will start with `Qwen/Qwen2.5-3B`, train it for 2 hours on normalized PubMedQA examples, and keep a separate PubMedQA test set untouched for the final showdown.

These are just example choices. You can swap in your own base model and your own instruction dataset when you are ready.

Paste your Gradients API key into the cell below. Everything else is already filled in.

In [ ]:
import os
import gradientsio as gradients

os.environ["GRADIENTS_API_KEY"] = "paste-your-api-key-here"
client = gradients.GradientsClient()

## Step 3: Send The Model To Training

This is the zero-faff part: we point Gradients at the medical training dataset, pick the base model, and launch the job.

The model and dataset below are examples. Replace them with your own Hugging Face model or dataset to train on your own data.

The printed task ID is your receipt. If you close the notebook, paste that ID into the next step and continue where you left off.

In [ ]:
task = client.train(
    model="Qwen/Qwen2.5-3B",
    task_type=gradients.TaskType.INSTRUCT,
    hours=2,
    dataset="gradients-io-tournaments/PubMedQA-Normalized-Train",
    field_instruction="instruction",
    field_input="input",
    field_output="output"
)

task_id = task.task_id
print(f"Training task created: {task_id}")

## Step 4: Wait For The Trained Model

Gradients now does the heavy lifting: dataset prep, scheduling, training, evaluation, and publishing the trained model repo.

Run this cell after launch. If you already have a task ID from an earlier run, paste it in and the notebook will wait for the final trained model.

In [ ]:
trained_model_repo = client.tasks.handle(task_id).wait().trained_model_repository

## Step 5: Pick Questions The Model Has Never Seen

Now we pull a few examples from the held-out test set. These are not part of the training run.

Use the PubMedQA test set here, or swap in your own held-out dataset to check how your model behaves on fresh examples.

This is where the story gets interesting: can a small base model learn the medical answer style from your custom data, then apply it to fresh biomedical questions?

In [ ]:
from IPython.display import Markdown, display

samples = gradients.load_dataset_rows("gradients-io-tournaments/PubMedQA-Normalized-Test")

def question(row):
    return (row.get("instruction") or "").strip()


def build_prompt(row):
    return f"{question(row)}\n\nAnswer:"


def format_result(index, row, base_answer, trained_answer):
    return f"""
---
### Example {index}

**Question**

{question(row)}

**Expected answer**

{(row.get('output') or '').strip()}

**Trained model**

{trained_answer.strip()}

**Base model**

{base_answer.strip()}
"""


for index, row in enumerate(samples, start=1):
    print(f"Example {index}: {question(row)}")

## Step 6: Watch The Before And After

Time for the proof.

We ask the original base model and the newly trained model the same unseen medical questions. If you changed the model or dataset above, use the same choices here.

Then we place both answers next to the expected answer so the improvement is easy to judge at a glance.

This is the whole Gradients loop: bring your data, train your model, test the difference.

In [ ]:
samples = gradients.load_dataset_rows("gradients-io-tournaments/PubMedQA-Normalized-Test")
prompts = [build_prompt(row) for row in samples]

sampler = gradients.ModelSampler()
base = sampler.generate("Qwen/Qwen2.5-3B", prompts)
trained = sampler.generate_with_adapter(
    trained_model_repo,
    prompts,
    base_model_repo="Qwen/Qwen2.5-3B",
)

for index, (row, base_answer, trained_answer) in enumerate(zip(samples, base, trained), start=1):
    display(Markdown(format_result(index, row, base_answer, trained_answer)))